# Ticket Deduction Data Pipeline

## Business purpose
Consolidate operational tickets and deduction-review information into a consistent, accessible Excel dataset. This public reconstruction preserves the reviewed notebook's reporting schema and confirmed business rules, without any charts.

## Report grain and responsibility
Repeated ticket rows are intentional. Each receives its ticket-level aggregated C&B amount; the sum is therefore a **report-grain total**, not a unique-ticket total. Every non-RMDyP review section, including blank or unknown, belongs to C&B by report convention. Direct deductions remain standalone records.

Responsibility assignments are supplied by a separate organizational process. This pipeline preserves them without determining or validating responsibility; conflicting assignments remain unresolved.

## Monetary quality
Missing and invalid amounts are distinct from genuine zeros. For legacy-report compatibility, they contribute an explicitly audited zero, with a quality-review flag. This does not establish that a deduction is absent or that an applicability conclusion is valid.

## How to use
Install pandas, numpy, openpyxl and xlsxwriter. Run all cells in order. Default mode uses independently generated fictional data. For authorized private use, disable demo mode and set input paths, sheets, header row and reporting period. Inputs remain unchanged.


## 1. Configuration
The reporting period is explicit. No company folders or Drive mounting are included.

In [ ]:
import os, re, unicodedata
from decimal import Decimal, InvalidOperation
import pandas as pd
import numpy as np
from IPython.display import display

DEMO_MODE = True
# Explicit reporting period; never inferred from unrelated closing dates.
REPORT_PERIOD = "2026-08"
TICKETS_PATH = "private_data/tickets.xlsx"
REVIEWS_PATH = "private_data/reviews.xlsx"
TICKETS_SHEET = 0
REVIEWS_SHEET = 0
HEADER_ROW = 1
OUTPUT_PATH = "results/ticket_deductions.xlsx"


## 2. Inputs
One ticket source and one review source. Keep their original records for audit; repeated ticket identifiers are not deduplicated.

In [ ]:
if DEMO_MODE:
    df = pd.DataFrame({
        "Ticket#": ["DEMO-001", "DEMO-001", "DEMO-002", "DEMO-003"],
        "Título": ["Sample task A", "Sample task A", "Sample task B", "Sample task C"],
        "Servicio": ["Service 1", "Service 1", "Service 2", "Service 2"],
        "Origen": ["Sample origin"] * 4,
        "Deducción RMDYP": ["$100.00", "$100.00", "", "invalid amount"],
        "Aplicabilidad": ["NO", "NO", "SI", "NO"],
    })
    df_cons = pd.DataFrame({
        "Ticket": ["DEMO-001", "DEMO-001", "DEMO-002", "DEMO-003", "Deducción Directa"],
        "Titulo": ["Sample task A", "Sample task A", "Sample task B", "Sample task C", "Sample adjustment"],
        "Servicio": ["Service 1", "Service 1", "Service 2", "Service 2", "Service 1"],
        "Origen": ["Sample origin"] * 5,
        "Apartado": ["R.M.D.y.P", "Review", "", "Unknown", "Review"],
        "Monto Aceptado": ["$100.00", "$25.00", "0", "invalid amount", "$10.00"],
        "Responsable": ["Organization A", "Organization B", "", "Organization A", "Organization B"],
    })
else:
    df = pd.read_excel(TICKETS_PATH, sheet_name=TICKETS_SHEET, header=HEADER_ROW, dtype=object)
    df_cons = pd.read_excel(REVIEWS_PATH, sheet_name=REVIEWS_SHEET, header=HEADER_ROW, dtype=object)
df = df.copy().reset_index(drop=True)
df_cons = df_cons.copy().reset_index(drop=True)
source_tickets = df.copy()
source_reviews = df_cons.copy()
# Source row locators are within the input table, independent of Excel header offset.
source_tickets.insert(0, "source_row", range(1, len(df) + 1))
source_reviews.insert(0, "source_row", range(1, len(df_cons) + 1))


## 3. Monetary parsing
Values are parsed to integer cents. Only declared decimal-point/thousands-comma formats are accepted. Every substitution records the original value, source and reason.

In [ ]:
# US/MX-style decimal point and comma thousands separators only.
# Decimal-comma inputs are rejected, not silently reinterpreted.
money_audit = []
def parse_money(value):
    if pd.isna(value) or (isinstance(value, str) and not value.strip()):
        return None, "MISSING", "No amount supplied"
    if isinstance(value, bool):
        return None, "INVALID", "Boolean is not an amount"
    text = str(value).strip()
    negative = text.startswith("(") and text.endswith(")")
    if negative:
        text = text[1:-1].strip()
    text = re.sub(r"^(?:MXN\s*|\$\s*)", "", text, flags=re.I)
    pattern = r"[+-]?(?:\d+|\d{1,3}(?:,\d{3})+)(?:\.\d{1,2})?"
    if not re.fullmatch(pattern, text):
        return None, "INVALID", "Unsupported amount format"
    try:
        amount = Decimal(text.replace(",", ""))
    except InvalidOperation:
        return None, "INVALID", "Amount cannot be parsed"
    if negative:
        if amount < 0:
            return None, "INVALID", "Conflicting negative signs"
        amount = -amount
    if not amount.is_finite():
        return None, "INVALID", "Non-finite amount"
    cents = int(amount * 100)
    return cents, "ZERO" if cents == 0 else "VALID", ""

def convert_amounts(series, source, ticket_series, policy="EXPLICIT_ZERO"):
    converted, statuses = [], []
    for row, (value, ticket) in enumerate(zip(series.tolist(), ticket_series.tolist()), start=1):
        cents, status, reason = parse_money(value)
        action = "PARSED"
        if cents is None:
            if policy != "EXPLICIT_ZERO":
                raise ValueError(f"{source} row {row}: {reason}")
            cents, action = 0, "SUBSTITUTED_ZERO_FOR_REPORT_COMPATIBILITY"
        converted.append(cents)
        statuses.append(status)
        money_audit.append({
            "source": source, "source_row": row, "ticket_original": ticket,
            "original_value": value, "parse_status": status, "reason": reason,
            "action": action, "report_amount": cents / 100,
        })
    return pd.Series(converted, index=series.index, dtype="int64"), pd.Series(statuses, index=series.index)


## 4. Reporting schema
These ordered fields and aliases come from the reviewed original notebook. Optional absent fields remain empty and are listed in the schema audit; required missing fields stop processing.

In [ ]:
# ==========================================
# BLOQUE: Construir df_final con columnas EXACTAS y en ORDEN dado
# - Toma de df las columnas existentes (acepta equivalencias)
# - Si falta alguna, la crea vacía (NaN)
# - Mantiene exactamente el orden solicitado
# ==========================================
import re, unicodedata
import pandas as pd
import numpy as np

def _strip_accents(s: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFD", str(s)) if unicodedata.category(c) != "Mn")

def _norm(s: str) -> str:
    s = _strip_accents(str(s)).lower()
    s = s.replace("\n"," ").replace("\r"," ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

# === 1) Lista de columnas destino EXACTAS (en el orden solicitado) ===
target_cols = [
    "Ticket#", "Título", "Origen", "Servicio", "Unidad Funcional", "Espacio",
    "Tipo de Espacio", "Tipo de Evento", "Tipo de Unidad Funcional", "caso al que hace referencia",
    "Estado", "Indicador Incumplido", "Factor de Falla", "Creado", "Fecha de Cierre",
    "Tiempo acordado para la Rectificación", "Hora y Fecha de Rectificación del Servicio",
    "Hora y Fecha de Aceptación del Tiempo Acordado", "Hora de Arribo",
    "Ponderación de Unidad Funcional", "Ponderación por Espacio", "Ponderación por Servicio",
    "tiempo de solución provisional en minutos (TSP)", "Tiempo acordado en minuto (TA)",
    "Tiempo de rectificacion en minutos (TRC)", "Tiempo de gracia o respuesta",
    "Resultado de tiempo de cierre", "Número de Turnos", "Número de Días Hábiles",
    " Deduccion Inicial Resultante ", "Nota de cierre", "Tipo", "Factor Curva de Aprendizaje",
    " Deducción Directa ", " Deducción Tipo 1 ", " Deducción tipo 2 ", " Deducción tipo 3 ",
    " Total Deducción ", " DEDUCCIÓN RMDYP ", "DIFERENCIA", "Fecha y Hora de Inicio del Servicio ",
    " Tiempo  para la realización del Servicio en minutos ", "Aplicabilidad",
    "Notas al Supervisor", "Tiempo de la programación"
]

# === 2) Equivalencias (cada destino acepta cualquiera de estas variantes en df) ===
equivalences = {
    "Título": ["Título", "Titulo", "Título del caso", "Titulo del caso"],
    "Creado": ["Creado", "Fecha de Apertura"],
    " Deduccion Inicial Resultante ": [" Deduccion Inicial Resultante ", "Deduccion Inicial Resultante", "Deducción Inicial Resultante", "Deduccion Inicial"],
    "Nota de cierre": ["Nota de cierre", "Nota de Cierre", "Nota de Seguimiento o Cierre", "Nota de  Seguimiento o cierre"],
    " Total Deducción ": [" Total Deducción ", "Total Deducción", "Total deduccion Inicial", "Total Deducción Inicial", "Total de deduccion Inicial"],
    " DEDUCCIÓN RMDYP ": [" DEDUCCIÓN RMDYP ", "DEDUCCIÓN RMDYP", "Deducción RMDYP", "Deduccion RMDYP", "Deducción Final Aplicable al Mes"],
    "DIFERENCIA": ["DIFERENCIA", "Diferencia", "Diferencia Deduccion Inicial vs. Final", "Diferencia Deducción Inicial vs. Final"],
    "Notas al Supervisor": ["Notas al Supervisor", "Notas del Supervisor"],
    "Tiempo de la programación": ["Tiempo de la programación", "Tiempo de la programacion", "Fecha fin de la programacion", "Fecha fin de la programación"],
    # El resto se buscan por nombre exacto (normalizado)
}



## 5. Consolidation and quality controls
Build report rows, enrich deductions, retain manual assignments, append direct adjustments and reconcile intentional repeated contributions. Missing and invalid monetary inputs remain visible in the quality flags.

In [ ]:
def ticket_key(value):
    return "" if pd.isna(value) else re.sub(r"\s+", "", str(value)).upper()
def is_rmdyp(value):
    return re.sub(r"[^A-Za-z]", "", str(value)).upper() == "RMDYP"
def is_direct(value):
    return _norm(value) == "deduccion directa"

# Reject conflicting normalized headers rather than arbitrarily choosing one.
def column_index(frame):
    result = {}
    for column in frame.columns:
        key = _norm(column)
        if key in result:
            raise ValueError(f"Ambiguous normalized columns: {result[key]} / {column}")
        result[key] = column
    return result
norm2real = column_index(df)
column_index(df_cons)
for name in ["Ticket", "Apartado", "Monto Aceptado", "Responsable", "Titulo", "Origen", "Servicio"]:
    if name not in df_cons.columns:
        raise KeyError(f"Missing review field: {name}")
def find_source(target):
    matches = list(dict.fromkeys(norm2real[_norm(candidate)]
                   for candidate in equivalences.get(target, [target])
                   if _norm(candidate) in norm2real))
    if len(matches) > 1:
        raise ValueError(f"Multiple source columns for {target}: {matches}")
    return matches[0] if matches else norm2real.get(_norm(target))
required = ["Ticket#", "Título", "Servicio", " DEDUCCIÓN RMDYP ", "Aplicabilidad"]
for target in required:
    if find_source(target) is None:
        raise KeyError(f"Required ticket field missing: {target}")
data, schema_audit = {}, []
for target in target_cols:
    source = find_source(target)
    data[target] = df[source].copy() if source is not None else pd.Series(pd.NA, index=df.index)
    schema_audit.append({"target_column": target, "source_column": source,
                         "status": "MAPPED" if source is not None else "OPTIONAL_MISSING"})
df_final = pd.DataFrame(data, columns=target_cols)
df_final["record_type"] = "TICKET"
df_final["source_row"] = range(1, len(df_final) + 1)
df_final["MES"] = REPORT_PERIOD
df_final["ticket_key"] = df_final["Ticket#"].map(ticket_key)
if df_final["ticket_key"].eq("").any():
    raise ValueError("Empty ticket keys require explicit review.")
rmdyp_cents, rmdyp_status = convert_amounts(
    df_final[" DEDUCCIÓN RMDYP "], "tickets:RMDYP", df_final["Ticket#"])
df_final[" DEDUCCIÓN RMDYP "] = rmdyp_cents / 100
df_final["RMDYP_AMOUNT_STATUS"] = rmdyp_status
df_final["DEDUCE"] = np.where(rmdyp_cents > 0, "SI", "NO")
df_final["OBSERVADO"] = (rmdyp_cents > 0).astype(int)

reviews = df_cons.copy()
reviews["source_row"] = range(1, len(reviews) + 1)
reviews["ticket_key"] = reviews["Ticket"].map(ticket_key)
reviews["record_type"] = np.where(reviews["Ticket"].map(is_direct), "DIRECT", "TICKET")
if reviews.loc[reviews["record_type"].eq("TICKET"), "ticket_key"].eq("").any():
    raise ValueError("Empty review ticket keys require explicit review.")
reviews["amount_cents"], reviews["AMOUNT_STATUS"] = convert_amounts(
    reviews["Monto Aceptado"], "reviews:accepted_amount", reviews["Ticket"])
# Confirmed business rule: all non-RMDyP sections, including blank/unknown, are C&B.
reviews["deduction_family"] = np.where(reviews["Apartado"].map(is_rmdyp), "RMDYP", "CB")
cb_reviews = reviews.loc[(reviews["deduction_family"] == "CB") & (reviews["record_type"] == "TICKET")]
cb_amounts = cb_reviews.groupby("ticket_key")["amount_cents"].sum()
cb_status = cb_reviews.groupby("ticket_key")["AMOUNT_STATUS"].agg(
    lambda values: "SOURCE_ISSUES" if values.isin(["MISSING", "INVALID"]).any() else "PARSED")
df_final["CB_AMOUNT_STATUS"] = df_final["ticket_key"].map(cb_status).fillna("NO_MATCH")
cb_cents = df_final["ticket_key"].map(cb_amounts).fillna(0).astype("int64")
df_final["DEDUCCIÓN C&B"] = cb_cents / 100
df_final["DEDUCE EN SEGUNDA VUELTA"] = np.where(cb_cents > 0, "SI", "NO")
df_final["DEDUCCIÓN TOTAL"] = (rmdyp_cents + cb_cents) / 100
app = df_final["Aplicabilidad"].fillna("").astype(str).str.strip().str.upper()
df_final["ESTATUS DE NO APLICABILIDAD SEGUNDA VUELTA"] = np.select(
    [(app == "NO") & (cb_cents <= 0), (app == "NO") & (cb_cents > 0)],
    ["ACEPTADO", "RECHAZADO"], default="-")
# The above statuses reproduce report rules; zero substitutions do NOT confirm evidence.
df_final["QUALITY_REVIEW_REQUIRED"] = (
    df_final["RMDYP_AMOUNT_STATUS"].isin(["MISSING", "INVALID"]) |
    df_final["CB_AMOUNT_STATUS"].eq("SOURCE_ISSUES"))

# Preserve all responsibility assignments supplied by the separate process.
responsibility_log = reviews[["source_row", "Ticket", "ticket_key", "Responsable", "record_type"]].copy()
responsibility_log["assignment_method"] = "EXTERNAL_PROCESS_SUPPLIED"
def unique_responsibles(values):
    return list(dict.fromkeys(str(value).strip() for value in values
                             if pd.notna(value) and str(value).strip()))
manual = responsibility_log.loc[responsibility_log["record_type"] == "TICKET"]
assignments = manual.groupby("ticket_key")["Responsable"].agg(unique_responsibles)
df_final["Responsable"] = df_final["ticket_key"].map(
    assignments.map(lambda names: names[0] if len(names) == 1 else pd.NA))
df_final["RESPONSIBILITY_STATUS"] = df_final["ticket_key"].map(
    assignments.map(lambda names: "MISSING" if not names else "CONFLICT" if len(names) > 1 else "MANUAL_SUPPLIED")
).fillna("MISSING")
responsibility_log["assignment_status"] = responsibility_log["ticket_key"].map(
    assignments.map(lambda names: "MISSING" if not names else "CONFLICT" if len(names) > 1 else "MANUAL_SUPPLIED")
).fillna("DIRECT_ROW_MANUAL")

# Trace each contributing C&B review to each repeated report row, intentionally.
contributions = df_final[["source_row", "ticket_key"]].merge(
    cb_reviews[["source_row", "ticket_key", "amount_cents", "AMOUNT_STATUS"]],
    on="ticket_key", how="inner", suffixes=("_ticket", "_review"), validate="many_to_many")
counts = df_final.groupby("ticket_key").size()
reconciliation = cb_amounts.rename("source_cb_cents").to_frame()
reconciliation["report_occurrences"] = counts.reindex(reconciliation.index).fillna(0).astype(int)
reconciliation["expected_report_cb_cents"] = reconciliation["source_cb_cents"] * reconciliation["report_occurrences"]
reconciliation["actual_report_cb_cents"] = cb_cents.groupby(df_final["ticket_key"]).sum().reindex(
    reconciliation.index).fillna(0).astype(int)
if not reconciliation["expected_report_cb_cents"].equals(reconciliation["actual_report_cb_cents"]):
    raise ValueError("Report-grain C&B reconciliation failed.")
if len(df_final) != len(df):
    raise ValueError("Ticket row count changed.")

# Direct deductions are standalone source rows, not a shared join key.
direct = reviews.loc[reviews["record_type"] == "DIRECT"]
extra = []
for _, row in direct.iterrows():
    output = {column: pd.NA for column in df_final.columns}
    output.update({
        "Ticket#": row["Ticket"], "Título": row["Titulo"], "Origen": row["Origen"],
        "Servicio": row["Servicio"], "MES": REPORT_PERIOD, "record_type": "DIRECT",
        "source_row": row["source_row"], "ticket_key": row["ticket_key"],
        " DEDUCCIÓN RMDYP ": 0.0, "DEDUCCIÓN C&B": row["amount_cents"] / 100,
        "DEDUCCIÓN TOTAL": row["amount_cents"] / 100, "Responsable": row["Responsable"],
        "RESPONSIBILITY_STATUS": "MANUAL_SUPPLIED" if pd.notna(row["Responsable"]) and str(row["Responsable"]).strip() else "MISSING",
        "CB_AMOUNT_STATUS": row["AMOUNT_STATUS"], "RMDYP_AMOUNT_STATUS": "NOT_APPLICABLE",
        "DEDUCE": "NO", "DEDUCE EN SEGUNDA VUELTA": "SI", "OBSERVADO": 1,
        "QUALITY_REVIEW_REQUIRED": row["AMOUNT_STATUS"] in ["MISSING", "INVALID"],
    })
    extra.append(output)
if extra:
    df_final = pd.concat([df_final, pd.DataFrame(extra)], ignore_index=True)
df_final["EN BOLETÍN DE CALIDAD?"] = np.where(
    df_final["Origen"].fillna("").map(_norm).eq("boletin de calidad"), "SI", "NO")
money_audit_df = pd.DataFrame(money_audit)
incidents = money_audit_df.loc[money_audit_df["parse_status"].isin(["MISSING", "INVALID"])]
unmatched_reviews = reviews.loc[(reviews["record_type"] == "TICKET") & ~reviews["ticket_key"].isin(df_final.loc[df_final["record_type"] == "TICKET", "ticket_key"])]
display(df_final[["Ticket#", "Servicio", "DEDUCCIÓN TOTAL", "QUALITY_REVIEW_REQUIRED", "RESPONSIBILITY_STATUS"]])


## 6. Synthetic checks
Expected demo: five report rows, report total 260, three monetary incidents, and two unresolved responsibility conflicts on repeated DEMO-001 rows. These assertions are not real operational metrics.

In [ ]:
# Expected fictional results, not company metrics.
if DEMO_MODE:
    assert len(df_final) == 5
    assert df_final["DEDUCCIÓN TOTAL"].sum() == 260.0
    assert len(incidents) == 3
    assert df_final.loc[df_final["Ticket#"] == "DEMO-001", "RESPONSIBILITY_STATUS"].eq("CONFLICT").all()
    assert cb_reviews["ticket_key"].tolist() == ["DEMO-001", "DEMO-002", "DEMO-003"]
    assert parse_money("$1,234.50")[0] == 123450
    assert parse_money("1.234,50")[1] == "INVALID"
    assert parse_money("0")[1] == "ZERO"
    assert parse_money("")[1] == "MISSING"
    assert parse_money("abc100")[1] == "INVALID"
    print("Synthetic assertions passed in this execution.")


## 7. Excel delivery
Includes report, monetary audit/incidents, source inputs, manual assignment history, contribution mapping, reconciliation, unmatched reviews and schema audit. These trace data transformations only—not the separate assignment process.

In [ ]:
os.makedirs(os.path.dirname(OUTPUT_PATH) or ".", exist_ok=True)
sheets = {
    "Report": df_final,
    "Monetary audit": money_audit_df,
    "Monetary incidents": incidents,
    "Responsibility assignments": responsibility_log,
    "CB contributions": contributions,
    "Reconciliation": reconciliation.reset_index(),
    "Unmatched reviews": unmatched_reviews,
    "Schema audit": pd.DataFrame(schema_audit),
    "Source tickets": source_tickets,
    "Source reviews": source_reviews,
}
# Disable formula interpretation for source text (Excel injection protection).
with pd.ExcelWriter(OUTPUT_PATH, engine="xlsxwriter",
                    engine_kwargs={"options": {"strings_to_formulas": False, "strings_to_urls": False}}) as writer:
    for name, frame in sheets.items():
        frame.to_excel(writer, sheet_name=name, index=False)
        sheet = writer.sheets[name]
        sheet.freeze_panes(1, 0)
        if len(frame.columns):
            sheet.set_column(0, len(frame.columns) - 1, 22)
            sheet.autofilter(0, 0, len(frame), len(frame.columns) - 1)
# Read back row count and amount totals from the written workbook.
written = pd.read_excel(OUTPUT_PATH, sheet_name="Report")
if len(written) != len(df_final) or not np.isclose(
        written["DEDUCCIÓN TOTAL"].sum(), df_final["DEDUCCIÓN TOTAL"].sum(), atol=0.005, rtol=0):
    raise ValueError("Written workbook differs from the report.")
print("Workbook:", os.path.abspath(OUTPUT_PATH))
print("Monetary incidents requiring review:", len(incidents))
print("Totals are report-grain totals, not unique-ticket totals.")


## Limitations and interpretation
Python execution and visual workbook inspection have not been completed by the authoring assistant. Run the synthetic checks before relying on the workbook. This is a portfolio reconstruction, not a validated production release. Monetary incident amounts remain unknown even when the compatibility report uses zero. Negative values are preserved, but positive-only deduction flags follow the original report convention. Conflicting responsibility assignments require review in the separate process. No real tickets, company amounts, responsible organizations or notebook outputs are published.